# UMAP Visualization

Companion notebook for Section 7 of the lecture notes.

Objectives:

- understand UMAP as a neighborhood-graph embedding method;
- compare the effects of `n_neighbors` and `min_dist`;
- use `fit_transform` and, when available, `transform`;
- contrast UMAP with PCA and t-SNE;
- keep visualization and pipeline use conceptually separate.


In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (7, 4)
plt.rcParams['axes.grid'] = True


## 1. Optional dependency

UMAP is provided by the `umap-learn` package, not by scikit-learn. The notebook is written to keep running even when the package is missing.


In [ ]:
import os
os.environ.setdefault('NUMBA_CACHE_DIR', '/tmp/numba-cache')

try:
    import umap.umap_ as umap
    UMAP_AVAILABLE = True
except ImportError:
    umap = None
    UMAP_AVAILABLE = False

print('UMAP available:', UMAP_AVAILABLE)
if not UMAP_AVAILABLE:
    print('Install with: python3 -m pip install umap-learn')


In [ ]:
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.manifold import trustworthiness
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_all, y_all = load_digits(return_X_y=True)
rng = np.random.default_rng(42)
idx = rng.choice(len(X_all), size=700, replace=False)
X = X_all[idx]
y = y_all[idx]
X_scaled = StandardScaler().fit_transform(X)

print('Subset shape:', X_scaled.shape)


## 2. PCA reference plot

This gives a linear baseline before using a nonlinear neighborhood-preserving embedding.


In [ ]:
X_pca = PCA(n_components=2, random_state=42).fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(7, 6))
scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='tab10', s=12, alpha=0.8)
ax.set_title('PCA baseline')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
plt.colorbar(scatter, ax=ax, label='digit')
plt.show()

print('PCA trustworthiness:', round(trustworthiness(X_scaled, X_pca, n_neighbors=10), 3))


## 3. UMAP default embedding

UMAP constructs a fuzzy neighborhood graph in the original space and searches for a low-dimensional layout whose graph structure is similar.


In [ ]:
if UMAP_AVAILABLE:
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, metric='euclidean', random_state=42)
    X_umap = reducer.fit_transform(X_scaled)

    fig, ax = plt.subplots(figsize=(7, 6))
    scatter = ax.scatter(X_umap[:, 0], X_umap[:, 1], c=y, cmap='tab10', s=12, alpha=0.8)
    ax.set_title('UMAP default-style embedding')
    ax.set_xlabel('UMAP dimension 1')
    ax.set_ylabel('UMAP dimension 2')
    plt.colorbar(scatter, ax=ax, label='digit')
    plt.show()

    print('UMAP trustworthiness:', round(trustworthiness(X_scaled, X_umap, n_neighbors=10), 3))
else:
    print('Skipping UMAP execution because umap-learn is not installed.')


## 4. `n_neighbors`: local versus broader structure

Small values focus on fine local neighborhoods. Larger values produce a more globally connected layout.


In [ ]:
if UMAP_AVAILABLE:
    neighbors_grid = [5, 15, 50]
    fig, axes = plt.subplots(1, len(neighbors_grid), figsize=(15, 4.5))
    for ax, n_neighbors in zip(axes, neighbors_grid):
        emb = umap.UMAP(
            n_neighbors=n_neighbors,
            min_dist=0.1,
            n_components=2,
            random_state=42,
        ).fit_transform(X_scaled)
        score = trustworthiness(X_scaled, emb, n_neighbors=10)
        ax.scatter(emb[:, 0], emb[:, 1], c=y, cmap='tab10', s=9, alpha=0.8)
        ax.set_title(f'n_neighbors={n_neighbors}\ntrust={score:.3f}')
        ax.set_xticks([])
        ax.set_yticks([])
    plt.tight_layout()
    plt.show()
else:
    print('Skipping hyperparameter grid because umap-learn is not installed.')


## 5. `min_dist`: compact versus spread-out clusters

Small values allow tight clusters. Larger values spread points more evenly in the embedding.


In [ ]:
if UMAP_AVAILABLE:
    min_dist_grid = [0.0, 0.1, 0.5]
    fig, axes = plt.subplots(1, len(min_dist_grid), figsize=(15, 4.5))
    for ax, min_dist in zip(axes, min_dist_grid):
        emb = umap.UMAP(
            n_neighbors=15,
            min_dist=min_dist,
            n_components=2,
            random_state=42,
        ).fit_transform(X_scaled)
        score = trustworthiness(X_scaled, emb, n_neighbors=10)
        ax.scatter(emb[:, 0], emb[:, 1], c=y, cmap='tab10', s=9, alpha=0.8)
        ax.set_title(f'min_dist={min_dist}\ntrust={score:.3f}')
        ax.set_xticks([])
        ax.set_yticks([])
    plt.tight_layout()
    plt.show()
else:
    print('Skipping hyperparameter grid because umap-learn is not installed.')


## 6. Transforming new data

Many UMAP implementations provide `transform`, which maps new observations into an existing fitted embedding. This is one practical difference from standard t-SNE.


In [ ]:
if UMAP_AVAILABLE:
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.25, stratify=y, random_state=42)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    X_train_umap = reducer.fit_transform(X_train)
    X_test_umap = reducer.transform(X_test)

    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(X_train_umap[:, 0], X_train_umap[:, 1], c='lightgray', s=12, label='train')
    scatter = ax.scatter(X_test_umap[:, 0], X_test_umap[:, 1], c=y_test, cmap='tab10', s=18, label='test')
    ax.set_title('Transforming held-out data with fitted UMAP')
    ax.legend()
    plt.colorbar(scatter, ax=ax, label='test digit')
    plt.show()
else:
    print('Skipping transform demo because umap-learn is not installed.')


## Exercises

1. Try `metric='manhattan'`. Which labels move or mix differently?
2. Increase `n_neighbors` to 100. Does the plot look more global or more local?
3. If using UMAP before a classifier, where should the UMAP step be placed to avoid leakage?


## Takeaways

- UMAP is nonlinear and neighborhood-preserving.
- `n_neighbors` controls locality; `min_dist` controls how tightly points pack.
- UMAP plots can show useful organization, but faraway cluster distances still require caution.
- If used as preprocessing, UMAP must be fit only on training data or inside a pipeline/CV fold.
